In [1]:
import importlib
import datetime
import json
from pathlib import Path

import pandas as pd
import numpy as np

import lib
importlib.reload(lib)

log = lib.getLogger('yelp_dataset')
working_directory = Path().cwd()
export_directory = working_directory / 'yelp_dataset'
export_clean_directory = export_directory / 'clean'

In [2]:
path_review_text = '/Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/yelp/yelp_dataset/clean/yelp_review_text_CLEAN.csv'

In [3]:
df_review_text = lib.read_data(path_review_text)

KeyboardInterrupt: 

### Sentiment Analysis

In [36]:
text = df_review_text['text'].tolist()

In [37]:
len(text)

6990280

In [47]:
import logging
logging.getLogger().setLevel(logging.INFO)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm import tqdm

model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name, truncation=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

roberta_base_remap = {
    'LABEL_2': 2,# 'Positive',
    'LABEL_1': 1,# 'Neutral',
    'LABEL_0': 0,# 'Negative'
}

sentiment = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    truncation=True,
    batch_size=32,
    padding="max_length",   # <—— makes everything exactly 512
    max_length=512,         # <—— hard limit
    device=0  # GPU = 0, CPU = -1
)

def batch_iter(lst, n=2000):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def batches(lst, n=2000):
    return list(batch_iter(lst, n))

results = []

count = 0
for batch in tqdm(batches(text, 2000)):
    out = sentiment(batch)
    sentiment_data = []

    for item in out:
        result = {
            'review_id': df_review_CLEAN.loc[count, 'review_id'],
            'sentiment': roberta_base_remap[item['label']],
            'score': item['score']
        }
        sentiment_data.append(result)
        count += 1

    with open(f'sentiment_results_{count}.json', 'a') as f:
        json.dump(sentiment_data, f)


Device set to use mps:0
  3%|▎         | 106/3496 [8:12:52<470:11:46, 499.32s/it]